# PSOLA-isolated Yorùbá TONE listening test — A/B rebuild (thin runner)

The **rigorous successor** to nb15's pilot, rebuilt as an **A/B forced choice**. The prior PSOLA kit's flips were
**inaudible**: it shifted F0 only inside the ~20–60 ms MMS-CTC sliver, not the voiced syllable rhyme (proof: 6/14
pairs had byte-identical `tone_i2`). **The fix:** grow the flip window to the **whole voiced rhyme**, **RESET** its
contour to the opposite tone band, process the correct twin **identically** (same artifact), and enforce a **hard
salience self-check** (realized ΔF0 ≥ 3 st on the re-extracted audio, residual past the band, class flipped) — any
clip whose flip didn't land is rejected at build time.

Each trial shows the written sentence + **two clips A and B** of the SAME utterance/voice/artifact (correct twin +
flipped twin, side randomized); the reviewer picks which sounds like correct Yorùbá. Ground truth is known by
construction. All logic lives in `pilot/build_psola_form.py` + `pilot/score_psola.py` (reusing
`build_pilot_form.py`'s S3/HTML/`tone_i2` machinery and `tone_metric`'s PSOLA primitives). This notebook:
1. installs deps + clones the repo,
2. builds → `pilot/psola_form.html` (self-contained, blind A/B) + `pilot/keymap_psola.json` (withheld GT),
3. prints the **automated salience table** + plays correct-vs-flipped pairs inline (§5), gated by `CONFIRM_FLIPS_AUDIBLE` (§5b),
4. (optional) scores a reviewer's pasted `PILOT2` answers.

**GPU:** none needed, but the MMS-yor aligner is slow on CPU — a **T4/L4** makes the build much faster.

## 1. Install + restart (numpy<2, scoring stack, parselmouth, tone-metric, ffmpeg)

In [ ]:
%pip install --quiet "numpy<2"
%pip install --quiet boto3 soundfile soxr librosa speechbrain "swift-f0" pyworld praat-parselmouth transformers safetensors "huggingface_hub>=0.24.0" torchaudio tqdm
%pip install --quiet --no-deps "git+https://github.com/mosesdaudu001/tone-on-a-budget.git"
%pip uninstall -y hf-xet
import subprocess; subprocess.run(["apt-get","-qq","install","-y","ffmpeg"], check=False)
import os
print("Installs done. RESTARTING so numpy<2 takes effect — run from the NEXT cell.", flush=True)
os._exit(0)

In [ ]:
import numpy as np
assert np.__version__.startswith("1."), "numpy<2 pin did not take — re-run install + restart"
import os, shutil, subprocess
REPO = "/content/tone-on-a-budget" if os.path.isdir("/content") else "/tmp/tone-on-a-budget"
if os.path.isdir(REPO): shutil.rmtree(REPO)
subprocess.run(["git","clone","--depth","1","https://github.com/mosesdaudu001/tone-on-a-budget.git",REPO], check=True)
PILOT = os.path.join(REPO, "pilot")
assert os.path.exists(os.path.join(PILOT,"build_psola_form.py")), PILOT
# sanity: parselmouth must import (the PSOLA engine)
import parselmouth; print("parselmouth", parselmouth.VERSION if hasattr(parselmouth,'VERSION') else 'ok', "| pilot dir:", PILOT)

## 2. Secrets → env (so the build script's `_secret` finds them without prompting)

In [ ]:
import os, getpass
def _secret(k):
    try:
        from google.colab import userdata
        v = userdata.get(k)
        if v: return v
    except Exception: pass
    return os.environ.get(k) or getpass.getpass(f"{k}: ")
os.environ["AWS_ACCESS_KEY_ID"] = _secret("AWS_ACCESS_KEY_ID")
os.environ["AWS_SECRET_ACCESS_KEY"] = _secret("AWS_SECRET_ACCESS_KEY")
os.environ["AWS_DEFAULT_REGION"] = os.environ.get("AWS_DEFAULT_REGION", "us-east-1")
_hf = _secret("HF_TOKEN"); os.environ["HF_TOKEN"] = _hf or ""
if _hf:
    from huggingface_hub import login; login(token=_hf)
os.environ["HF_HUB_DISABLE_XET"] = "1"
print("secrets in env.")

## 3. (optional) DRY RUN — list the native clips it will consider; manipulate / score NOTHING

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, os.path.join(PILOT,"build_psola_form.py"),
                "--dry-run", "--n-clips","24","--n-catch","5"],
               check=True, env=os.environ.copy())

## 4. BUILD — download native, grow the whole-rhyme flip, score tone_i2 (frozen + deployed), embed A/B audio

Writes `pilot/psola_form.html` (blind **A/B** form) + `pilot/keymap_psola.json` (withheld) **inside the cloned repo**.
Each trial is an artifact-matched **PAIR** of the SAME sentence — the correct twin and the flipped twin — with the
correct side randomized. Every shipped flip is RESET to the opposite tone band over the **whole voiced rhyme** and
must pass the **hard salience self-check** (realized ΔF0 ≥ 3 st on the re-extracted audio, residual past the band,
class flipped) or the clip is rejected. `--n-clips 24` real pairs + `--n-catch 5` catch.

In [ ]:
import subprocess, sys
OUT = PILOT
subprocess.run([sys.executable, os.path.join(PILOT,"build_psola_form.py"),
                "--n-clips","24","--n-catch","5",
                "--seed","4242","--out-dir",OUT],
               check=True, env=os.environ.copy())
HTML = os.path.join(OUT,"psola_form.html"); KEYMAP = os.path.join(OUT,"keymap_psola.json")
print("\nbuilt:", HTML, "(%.2f MB)" % (os.path.getsize(HTML)/1e6))

## 5. EAR-CHECK + automated SALIENCE TABLE (decisive gate)

For every shipped pair this prints the **automated salience audit** (realized ΔF0 in semitones, opposite-band
PASS/FAIL, pred-flip PASS/FAIL — all measured on the *synthesized* flipped audio at build time) **next to inline
`▶ correct` / `▶ flipped` players**. By construction every shipped pair already PASSED the self-check (failing
clips were rejected), so `ALL_SALIENT` should be `True` — the table makes that auditable, and the players let you
confirm by ear that each FLIPPED clip genuinely sounds like **WRONG TONE** (not a glitch / the same word). The
labels are shown here (unlike the blind form) so you can compare; the side shown as "CORRECT" is `side_map`.

In [ ]:
import json, re, base64, tempfile, IPython.display as ipd
html = open(HTML, encoding="utf-8").read()
ITEMS = {it["item_id"]: it for it in json.loads(re.search(r"const ITEMS = (\[.*?\]);", html, re.S).group(1))}
KEY = json.load(open(KEYMAP, encoding="utf-8"))

# automated salience table over every shipped (non-catch) PAIR
MIN_REALIZED_ST = 3.0
rows = [(iid, v) for iid, v in KEY.items() if not v["is_catch"]]
print(f"{'item':>7} {'flip':>14} {'ΔF0(st)':>8} {'band':>5} {'pred-flip':>9}   intended")
print("-" * 78)
all_pass = True
for iid, v in rows:
    rst = v.get("realized_st")
    delta_ok = (rst is not None and rst >= MIN_REALIZED_ST)
    band_ok = bool(v.get("band_ok"))
    pred_ok = bool(v.get("pred_flip_ok"))
    ok = delta_ok and band_ok and pred_ok
    all_pass = all_pass and ok
    print(f"{iid:>7} {str(v.get('flip_dir')):>14} {('%.2f'%rst) if rst is not None else '  n/a':>8} "
          f"{'PASS' if band_ok else 'FAIL':>5} {'PASS' if pred_ok else 'FAIL':>9}   {v['intended_text'][:42]}")
ALL_SALIENT = bool(all_pass and len(rows) > 0)
print("-" * 78)
print(f"ALL_SALIENT = {ALL_SALIENT}   ({len(rows)} pairs; every shipped flip passed the build-time self-check)")

# inline A/B players (labels revealed for the operator) — confirm each FLIPPED clip sounds like WRONG TONE
tmp = tempfile.mkdtemp()
def _play(uri, name, tag):
    header, b64 = uri.split(",", 1)
    ext = "mp3" if "mpeg" in header else "wav"
    fp = f"{tmp}/{name}.{ext}"; open(fp, "wb").write(base64.b64decode(b64))
    ipd.display(ipd.HTML(f"<b>{tag}</b>")); ipd.display(ipd.Audio(fp))
for iid, v in rows[:6]:
    it = ITEMS[iid]
    correct_side, flipped_side = v["side_map"], ("B" if v["side_map"] == "A" else "A")
    ipd.display(ipd.HTML(f"<hr><h4>{iid} — {v['pair_id']} — TBU #{v.get('tbu_index')}, flip {v.get('flip_dir')} "
                         f"(realized {v.get('realized_st')} st)</h4><div>{it['text']}</div>"))
    _play(it["audio" + correct_side], iid + "_corr", f"✓ CORRECT (side {correct_side})")
    _play(it["audio" + flipped_side], iid + "_flip", f"✗ FLIPPED  (side {flipped_side}) — must sound like WRONG TONE")
print(f"\nPlayed {min(6, len(rows))} pairs. ALL_SALIENT={ALL_SALIENT}. If each FLIPPED clip sounds like WRONG TONE, proceed to the gate below.")

## 5b. GATE — confirm the flips are audibly WRONG TONE (human-in-the-loop)

The export in §6 is blocked until you attest here. **Listen to every ✗ FLIPPED clip played in §5** and confirm
each sounds like **wrong tone** (not a glitch, and clearly different from its ✓ CORRECT twin). Only then set
`CONFIRM_FLIPS_AUDIBLE = True` in the cell below and re-run it. This is the deliberate ear-confirmation
checkpoint required by `PSOLA_RECIPE.md` §5 — `ALL_SALIENT` (automated) and `CONFIRM_FLIPS_AUDIBLE` (human)
must BOTH be True before the form can be downloaded/sent.

In [ ]:
# 5b. GATE — confirm by EAR before the form can be sent.
# Set CONFIRM_FLIPS_AUDIBLE = True ONLY after listening to EVERY ✗ FLIPPED clip played in §5 above and
# confirming each one sounds like WRONG TONE (not a glitch, and clearly different from its ✓ CORRECT twin).
CONFIRM_FLIPS_AUDIBLE = False  # <- flip to True only after you have listened and confirmed
assert ALL_SALIENT and CONFIRM_FLIPS_AUDIBLE, (
    "BLOCKED: confirm by ear in §5b first — set CONFIRM_FLIPS_AUDIBLE = True after listening to every "
    "FLIPPED clip in §5 and confirming it sounds like WRONG TONE (and ALL_SALIENT must be True).")
print("§5b PASSED: salience self-check True AND operator confirmed flips audibly wrong tone. Cleared to send.")

## 6. Download the form to send (Colab) — gated on §5b

Send `psola_form.html` to a native reviewer (an **A/B** form: two versions A and B of each sentence, pick which
sounds like correct Yorùbá). `keymap_psola.json` is the **withheld key — never send it.** The cell below re-asserts
the §5b gate, so it refuses to run until `ALL_SALIENT` and `CONFIRM_FLIPS_AUDIBLE` are both `True`.

In [ ]:
# SAFETY: the form cannot be saved/sent unless the automated self-check passed AND you confirmed by ear (§5b).
assert ALL_SALIENT and CONFIRM_FLIPS_AUDIBLE, (
    "BLOCKED: run §5 + §5b first — ALL_SALIENT and CONFIRM_FLIPS_AUDIBLE must both be True before sending.")
try:
    from google.colab import files; files.download(HTML)
except Exception:
    print("(not on Colab) the form is at:", HTML)

## 7. (optional) SCORE a reviewer's pasted answers (known ground truth)

Paste the reviewer's `PILOT2;item01=A;item02=B;...` code into `ANSWERS`. Run with no answers to get the **human-free
metric sensitivity** (FROZEN paired-win) alone. The keystone is **human %-correct on the A/B pairs vs the 50%
baseline**; agreement is **AUROC / point-biserial of the FROZEN tone_i2 margin vs human-correct**. Catch < 80% ⇒
the rater is discarded; < 24 scoreable pairs ⇒ "NO SCOREABLE PILOT".

In [ ]:
import subprocess, sys
ANSWERS = ""   # <- paste the reviewer's pasted A/B code here, e.g. "PILOT2;item01=A;item02=B;item03=unsure;..."
cmd = [sys.executable, os.path.join(PILOT,"score_psola.py"), "--keymap", KEYMAP]
if ANSWERS.strip():
    cmd += ["--answers", ANSWERS]
subprocess.run(cmd, check=True, env=os.environ.copy())